In [6]:
"""
Step 3a of 5 — build forecast_errors.csv from the archived vintages.

This is where the "one-year-ahead" convention is fixed. A vintage published in
year t is scored on its forecast for year t+1:

    round          IMF column      OECD edition          AMECO vintage   target
    autumn 2020    F2020           EO108 (Dec 2020)      autumn 2020     2021
    spring 2021    S2021           EO109 (May 2021)      spring 2021     2022
    autumn 2021    F2021           EO110 (Dec 2021)      autumn 2021     2022
    spring 2022    S2022           EO111 (Jun 2022)      spring 2022     2023
    autumn 2022    F2022           EO112 (Nov 2022)      autumn 2022     2023

Note that the horizon is NOT constant: an autumn vintage sees roughly 14 months
to the end of its target year, a spring vintage roughly 20. The paper should say
so. Outcomes are the latest revision (April 2026 WEO), the same basis for all
three institutions.

UPLOAD (any order, names only have to contain the tokens shown)
    WEOhistorical.xlsx                    IMF Historical WEO Forecasts Database
    WEOApr2026all.xlsx                    outcomes, sheet "Countries"
    OECD ... EO108 ... .csv  x5           editions 108-112, or one zip holding them
    ameco_autumn2020.zip ... x5           vintage zips, or extracted AMECO6.TXT
WRITES
    forecast_errors.csv    iso, round, target, IMF, OECD, EC, actual
"""
import glob
import os
import zipfile

import numpy as np
import pandas as pd

ROUNDS = [("autumn2020", "F2020", 108, 2021),
          ("spring2021", "S2021", 109, 2022),
          ("autumn2021", "F2021", 110, 2022),
          ("spring2022", "S2022", 111, 2023),
          ("autumn2022", "F2022", 112, 2023)]
AMECO_ISO_FIX = {"ROM": "ROU"}
WORK = "_vintages"

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def pick(tokens, exts):
    for f in sorted(glob.glob("*")):
        low = f.lower()
        if low.endswith(exts) and all(t in low for t in tokens):
            return f
    return None


# ---------------------------------------------------------------- unpack ----
def unpack():
    """Flatten every zip in the directory, including AMECO's nested ameco6.zip."""
    os.makedirs(WORK, exist_ok=True)
    for z in sorted(glob.glob("*.zip")):
        low = os.path.basename(z).lower()
        tag = next((v for v, _, _, _ in ROUNDS
                    if v in low or v in low.replace("_", "")), None)
        dest = os.path.join(WORK, tag) if tag else WORK
        os.makedirs(dest, exist_ok=True)
        try:
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dest)
        except zipfile.BadZipFile:
            print(f"  ! {z} is not a readable zip (truncated upload?)")
            continue
        for inner in glob.glob(os.path.join(dest, "**", "ameco6.zip"),
                               recursive=True):
            with zipfile.ZipFile(inner) as zf:
                zf.extractall(os.path.dirname(inner))


def find_ameco(vintage):
    for pat in (os.path.join(WORK, vintage, "**", "AMECO6.TXT"),
                os.path.join(WORK, "**", f"*{vintage}*", "**", "AMECO6.TXT")):
        hits = [h for h in glob.glob(pat, recursive=True) if "__MACOSX" not in h]
        if hits:
            return hits[0]
    return None


def find_oecd(edition):
    hits = [h for h in glob.glob("**/*.csv", recursive=True)
            if f"eo{edition}" in os.path.basename(h).lower()
            and "__MACOSX" not in h]
    return hits[0] if hits else None


def missing():
    out = []
    if pick(("weohistorical",), (".xlsx",)) is None:
        out.append("WEOhistorical.xlsx")
    if pick(("weo", "2026"), (".xlsx",)) is None and \
       pick(("weoapr",), (".xlsx",)) is None:
        out.append("WEOApr2026all.xlsx")
    out += [f"OECD EO{ed}" for _, _, ed, _ in ROUNDS if find_oecd(ed) is None]
    out += [f"AMECO {v}" for v, _, _, _ in ROUNDS if find_ameco(v) is None]
    return out


unpack()
gaps = missing()
for _ in range(6):                       # upload in as many batches as you like
    if not gaps or not IN_COLAB:
        break
    print(f"still needed ({len(gaps)}): {', '.join(gaps)}")
    print("Select some or all of them; you will be asked again if any remain.\n")
    files.upload()
    unpack()
    gaps = missing()
if gaps:
    raise FileNotFoundError(
        "still missing: " + ", ".join(gaps) +
        f"\nPresent: {sorted(glob.glob('*'))}")
print("all inputs present\n")


# ------------------------------------------------------------------- IMF ----
f_hist = pick(("weohistorical",), (".xlsx",))
f_out = pick(("weo", "2026"), (".xlsx",)) or pick(("weoapr",), (".xlsx",))

h = pd.read_excel(f_hist, sheet_name="ngdp_rpch")
h = h[h.ISOAlpha_3Code.notna()]
imf = {}
for _, vc, _, tgt in ROUNDS:
    col = f"{vc}ngdp_rpch"
    s = h.loc[h.year == tgt, ["ISOAlpha_3Code", col]].copy()
    s[col] = pd.to_numeric(s[col], errors="coerce")
    imf[vc] = s.dropna().set_index("ISOAlpha_3Code")[col]

# ------------------------------------------------------------------ OECD ----
oecd = {}
for _, _, ed, tgt in ROUNDS:
    f = find_oecd(ed)
    if f is None:
        raise FileNotFoundError(f"OECD edition EO{ed} not found")
    cols = pd.read_csv(f, nrows=0).columns
    area = "LOCATION" if "LOCATION" in cols else "REF_AREA"
    var = "VARIABLE" if "VARIABLE" in cols else "MEASURE"
    d = pd.read_csv(f, usecols=[area, var, "TIME_PERIOD", "OBS_VALUE"],
                    low_memory=False)
    d = d[(d[var] == "GDPV_ANNPCT") & (d.TIME_PERIOD == tgt)]
    oecd[ed] = d.set_index(area).OBS_VALUE

# -------------------------------------------------------------------- EC ----
ec = {}
for v, _, _, tgt in ROUNDS:
    f = find_ameco(v)
    if f is None:
        raise FileNotFoundError(f"AMECO vintage {v} not found under {WORK}/")
    a = pd.read_csv(f, sep=";", encoding="latin-1", low_memory=False)
    a.columns = [c.strip() for c in a.columns]
    a["CODE"] = a.CODE.astype(str).str.strip()
    ov = a[a.CODE.str.endswith(".OVGD")].copy()
    ov["iso"] = ov.CODE.str.split(".").str[0].replace(AMECO_ISO_FIX)
    g = {}
    for _, r in ov.iterrows():                   # AMECO is levels, not growth
        try:
            g[r.iso] = (float(r[str(tgt)]) / float(r[str(tgt - 1)]) - 1) * 100
        except Exception:
            pass
    ec[v] = pd.Series(g)

# -------------------------------------------------------------- outcomes ----
w = pd.read_excel(f_out, sheet_name="Countries")
w = w[w["INDICATOR.ID"] == "NGDP_RPCH"]
act = {y: pd.to_numeric(w.set_index("COUNTRY.ID")[y], errors="coerce")
       for y in sorted({t for _, _, _, t in ROUNDS})}

# ----------------------------------------------------------------- build ----
rows = []
for v, vc, ed, tgt in ROUNDS:
    isos = (set(imf[vc].index) & set(oecd[ed].index) & set(ec[v].index)
            & set(act[tgt].dropna().index))
    for iso in sorted(isos):
        rows.append({"iso": iso, "round": v, "target": tgt,
                     "IMF": imf[vc][iso], "OECD": oecd[ed][iso],
                     "EC": ec[v][iso], "actual": act[tgt][iso]})
    print(f"{v:<11} EO{ed} / {vc} -> target {tgt}: {len(isos):>3} economies")

e = pd.DataFrame(rows)
e.round(6).to_csv("forecast_errors.csv", index=False)

n = e.groupby("iso").size()
print(f"\nforecast_errors.csv: {len(e)} rows, {e.iso.nunique()} economies")
print(f"  with all 5 rounds: {(n == 5).sum()}")
short = sorted(n[n < 3].index)
if short:
    print(f"  fewer than 3 rounds (dropped in step 3): {', '.join(short)}")

for k in ["IMF", "OECD", "EC"]:
    e["s_" + k] = e[k] - e.actual
full = e[e.iso.isin(n[n >= 3].index)]
print("\n  panel MAE  " + " | ".join(
    f"{k} {full['s_'+k].abs().groupby(full.iso).mean().mean():.3f}"
    for k in ["IMF", "OECD", "EC"]))
print(f"  2021 target inflates these: mean |error| "
      f"{full[full.target==2021][['s_IMF','s_OECD','s_EC']].abs().mean().mean():.2f} "
      f"vs {full[full.target!=2021][['s_IMF','s_OECD','s_EC']].abs().mean().mean():.2f} "
      f"for 2022-2023")

if IN_COLAB:
    files.download("forecast_errors.csv")

still needed (12): WEOhistorical.xlsx, WEOApr2026all.xlsx, OECD EO108, OECD EO109, OECD EO110, OECD EO111, OECD EO112, AMECO autumn2020, AMECO spring2021, AMECO autumn2021, AMECO spring2022, AMECO autumn2022
Select some or all of them; you will be asked again if any remain.



Saving ameco_autumn2020.zip to ameco_autumn2020.zip
Saving ameco_autumn2021.zip to ameco_autumn2021.zip
Saving ameco_autumn2022.zip to ameco_autumn2022.zip
Saving ameco_spring2020.zip to ameco_spring2020.zip
Saving ameco_spring2021.zip to ameco_spring2021.zip
Saving OECD,DF_EO108_INTERNET,+..A.csv to OECD,DF_EO108_INTERNET,+..A.csv
Saving OECD,DF_EO109_INTERNET,+..A.csv to OECD,DF_EO109_INTERNET,+..A.csv
Saving OECD,DF_EO110_INTERNET,+..A.csv to OECD,DF_EO110_INTERNET,+..A.csv
Saving OECD,DF_EO111_INTERNET,+..A.csv to OECD,DF_EO111_INTERNET,+..A.csv
Saving OECD,DF_EO112_INTERNET,+..A.csv to OECD,DF_EO112_INTERNET,+..A.csv
Saving WEOApr2026all.xlsx to WEOApr2026all.xlsx
Saving WEOhistorical.xlsx to WEOhistorical.xlsx
still needed (1): AMECO spring2022
Select some or all of them; you will be asked again if any remain.



Saving ameco_spring2022.zip to ameco_spring2022.zip
all inputs present

autumn2020  EO108 / F2020 -> target 2021:  36 economies
spring2021  EO109 / S2021 -> target 2022:  36 economies
autumn2021  EO110 / F2021 -> target 2022:  36 economies
spring2022  EO111 / S2022 -> target 2023:  36 economies
autumn2022  EO112 / F2022 -> target 2023:  37 economies

forecast_errors.csv: 181 rows, 37 economies
  with all 5 rounds: 36
  fewer than 3 rounds (dropped in step 3): HRV

  panel MAE  IMF 1.804 | OECD 1.911 | EC 1.835
  2021 target inflates these: mean |error| 2.95 vs 1.58 for 2022-2023


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>